# Spotify — `songs_with_attributes_and_lyrics.csv` Preprocessing

Loads the raw Spotify dataset from `data/raw/`, applies all cleaning and deduplication steps for a **lyrics-based unsupervised recommendation system**, and saves the result to `data/processed/spotify_clean.csv`.

**Constraints:** no NLP libraries · no stemming/lemmatisation · no stopword removal · preserve lyric structure (line breaks).

In [1]:
# Bring in the tools we need:
# re        -> for finding and replacing patterns in text (like removing punctuation)
# hashlib   -> for creating a unique fingerprint from lyrics text
# Path      -> makes file/folder paths easy to work with on any OS
# pandas    -> the main library for loading and working with spreadsheet-like data
import re
import hashlib
from pathlib import Path
import pandas as pd

In [ ]:
# ── Paths — locate the project root (replace deprecated 'approot') ─────────────────
# Rules:
# - If running inside the project's .venv, go up one level and use that as the candidate.
# - Otherwise search upward for repository markers ('.git' or README files).
# - If not found, fall back to the current working directory.
from pathlib import Path

cwd = Path.cwd()
if cwd.name == '.venv':
    base = cwd.parent
else:
    base = cwd

# Detect repository root by looking for common markers
APP_ROOT = None
p = base
for _ in range(10):
    if (p / '.git').exists() or (p / 'readme.md').exists() or (p / 'README.md').exists():
        APP_ROOT = p.resolve()
        break
    if p.parent == p:
        break
    p = p.parent

# final fallback: use base (project root candidate)
if APP_ROOT is None:
    APP_ROOT = base.resolve()

RAW_CSV = APP_ROOT / 'data' / 'raw' / 'songs_with_attributes_and_lyrics.csv'
PROCESSED_DIR = APP_ROOT / 'data' / 'processed'
OUT_CSV = PROCESSED_DIR / 'spotify_clean.csv'

print('app root:   ', APP_ROOT)
print('raw exists?  ', RAW_CSV.exists())
print('output will be:', OUT_CSV)

app root:    C:\Users\Aman Tewari\Documents\aaa-college\sem6\SLA-pvt\approot
raw exists?   True
output will be: C:\Users\Aman Tewari\Documents\aaa-college\sem6\SLA-pvt\approot\data\processed\spotify_clean.csv


In [3]:
# Reusable helper functions
# These are small tools we'll call repeatedly during cleaning.

def ascii_ratio(text: str) -> float:
    # Return fraction of characters that are ASCII (common in English text).
    if not isinstance(text, str) or len(text) == 0:
        return 0.0
    return sum(ord(c) < 128 for c in text) / len(text)


def normalize_meta(s: str) -> str:
    # Clean a title or artist string to a simple canonical form.
    if not isinstance(s, str):
        return ''
    s = s.lower()
    s = re.sub(r"\([^)]*(live|remaster(ed)?).*?\)", '', s, flags=re.IGNORECASE)
    s = re.sub(r"[\-\u2013\u2014]\s*live\s*$", '', s, flags=re.IGNORECASE)
    s = re.sub(r"\([^)]*\)", '', s)
    s = re.sub(r"[^\w\s]", ' ', s)
    return re.sub(r"\s+", ' ', s).strip()


def normalize_lyrics(text: str) -> str:
    # Lowercase, remove most punctuation but keep apostrophes, and normalize spaces per line.
    if not isinstance(text, str):
        return ''
    t = text.lower()
    t = re.sub(r"[^\w\s']", ' ', t)
    lines = [re.sub(r'[ \t]+', ' ', line).strip() for line in t.splitlines()]
    return '\n'.join(lines)


def md5_hash(s: str) -> str:
    # Fast fingerprint for deduplication
    return hashlib.md5(s.encode('utf-8')).hexdigest()

## Memory-light streaming run (two passes)

Run this section instead of the step-by-step cells below when you need lower memory/CPU use. It streams the CSV in chunks, computes the word-count bounds in pass 1, then writes the filtered, deduplicated output in pass 2. Chunks are only for processing; the final output is a single consolidated spotify_clean.csv containing every row that survives the filters and dedup.

In [4]:
# Streaming-friendly preprocessing (two passes)
# Pass 1: stream the CSV to compute word-count IQR bounds while tracking duplicates.
# Pass 2: stream again, apply dedup + IQR filter, and append to the output CSV.

CHUNK_SIZE = 50_000  # adjust smaller if memory is very tight

# Columns we need from the raw CSV
RAW_COLS = ['id', 'name', 'artists', 'lyrics']


def clean_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    """Apply all row-level cleaning/normalization steps used in the original pipeline."""
    chunk = chunk.rename(columns={'name': 'title', 'artists': 'artist'})
    chunk = chunk[chunk['lyrics'].notna()].copy()

    # Basic filters
    chunk['_word_count'] = chunk['lyrics'].astype(str).str.split().str.len()
    chunk = chunk[chunk['_word_count'] >= 30].copy()
    chunk['_ascii'] = chunk['lyrics'].astype(str).map(ascii_ratio)
    chunk = chunk[chunk['_ascii'] >= 0.90].copy()

    # Clean artist
    chunk['artist'] = chunk['artist'].astype(str).str.lower()
    chunk['artist'] = chunk['artist'].str.replace(r'\[|\]', '', regex=True)
    chunk['artist'] = chunk['artist'].str.replace("'", '', regex=False)
    chunk['artist'] = chunk['artist'].str.replace(r'\s+', ' ', regex=True).str.strip()

    # Titles
    chunk['title'] = chunk['title'].astype(str)
    chunk['normalized_title'] = chunk['title'].map(normalize_title_strong)
    chunk['normalized_artist'] = chunk['artist'].str.lower()
    chunk['normalized_artist'] = chunk['normalized_artist'].str.replace(r'[^A-Za-z0-9 ]+', ' ', regex=True)
    chunk['normalized_artist'] = chunk['normalized_artist'].str.replace(r'\s+', ' ', regex=True).str.strip()

    # Lyrics normalizations
    chunk['lyrics'] = chunk['lyrics'].astype(str).map(normalize_lyrics)
    chunk['normalized_lyrics'] = chunk['lyrics'].str.lower()
    chunk['normalized_lyrics'] = chunk['normalized_lyrics'].str.replace(r'[^\w\s]', ' ', regex=True)
    chunk['normalized_lyrics'] = chunk['normalized_lyrics'].str.replace(r'\s+', ' ', regex=True).str.strip()

    # Dedup helpers
    chunk['_comp_key'] = chunk['normalized_title'].fillna('') + '_' + chunk['normalized_artist'].fillna('')
    chunk['_lyrics_hash'] = chunk['lyrics'].map(md5_hash)
    return chunk


# ── Pass 1: compute IQR bounds with streaming dedup ─────────────────────────────
seen_comp_keys_pass1: set[str] = set()
seen_lyrics_hash_pass1: set[str] = set()
word_counts: list[int] = []
rows_in = rows_after_pass1 = 0

for i, chunk in enumerate(pd.read_csv(RAW_CSV, usecols=RAW_COLS, chunksize=CHUNK_SIZE, low_memory=False)):
    rows_in += len(chunk)
    chunk = clean_chunk(chunk)

    # Drop duplicates against what we've already seen in this pass
    mask_new = (~chunk['_comp_key'].isin(seen_comp_keys_pass1)) & (~chunk['_lyrics_hash'].isin(seen_lyrics_hash_pass1))
    chunk = chunk[mask_new]
    seen_comp_keys_pass1.update(chunk['_comp_key'])
    seen_lyrics_hash_pass1.update(chunk['_lyrics_hash'])

    word_counts.extend(chunk['_word_count'].astype(int).tolist())
    rows_after_pass1 += len(chunk)
    print(f'Pass1 chunk {i:03d}: kept {len(chunk):6d} / {rows_in:6d} so far')

if not word_counts:
    raise RuntimeError('No rows survived initial filtering; check the raw CSV and filters.')

wc_series = pd.Series(word_counts)
q1, q3 = wc_series.quantile(0.25), wc_series.quantile(0.75)
iqr = q3 - q1
lo = max(1, int(q1 - 1.5 * iqr))
hi = int(q3 + 1.5 * iqr)
print(f'Pass1 complete: input {rows_in:,} → kept {rows_after_pass1:,} (unique). Word-count IQR bounds [{lo}, {hi}].')


# ── Pass 2: stream, dedup, apply IQR bounds, and write ─────────────────────────
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
if OUT_CSV.exists():
    OUT_CSV.unlink()

seen_comp_keys_pass2: set[str] = set()
seen_lyrics_hash_pass2: set[str] = set()
rows_written = rows_after_pass2 = 0

for i, chunk in enumerate(pd.read_csv(RAW_CSV, usecols=RAW_COLS, chunksize=CHUNK_SIZE, low_memory=False)):
    chunk = clean_chunk(chunk)

    mask_new = (~chunk['_comp_key'].isin(seen_comp_keys_pass2)) & (~chunk['_lyrics_hash'].isin(seen_lyrics_hash_pass2))
    chunk = chunk[mask_new]
    seen_comp_keys_pass2.update(chunk['_comp_key'])
    seen_lyrics_hash_pass2.update(chunk['_lyrics_hash'])

    chunk = chunk[(chunk['_word_count'] >= lo) & (chunk['_word_count'] <= hi)].copy()

    out = chunk[['id', 'title', 'artist', 'normalized_title', 'normalized_artist', 'lyrics', 'normalized_lyrics']]
    mode = 'w' if rows_written == 0 else 'a'
    header = rows_written == 0
    out.to_csv(OUT_CSV, mode=mode, header=header, index=False)

    rows_written += len(out)
    rows_after_pass2 += len(chunk)
    print(f'Pass2 chunk {i:03d}: wrote {len(out):6d} (total written {rows_written:,})')

print(f'✔ Done. Final rows written: {rows_written:,} → {OUT_CSV}')


NameError: name 'normalize_title_strong' is not defined

In [ ]:
# ── Step 1 · Load the raw data
# Read the CSV file into a pandas DataFrame (think of it like an Excel table).
# We check the file exists first so we get a helpful error message instead of a cryptic crash.
if not RAW_CSV.exists():
    raise FileNotFoundError(
        f'Raw CSV not found at {RAW_CSV}\n'
        'Place songs_with_attributes_and_lyrics.csv inside data/raw/ and re-run.'
    )

# low_memory=False tells pandas to read the whole file before deciding column types
# — avoids misleading "mixed types" warnings on large files.
df = pd.read_csv(RAW_CSV, low_memory=False)
print(f'Loaded  : {len(df):,} rows  |  columns: {list(df.columns)}')

Loaded  : 955,320 rows  |  columns: ['id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics']


In [ ]:
# ── Step 2 · Drop columns we do not need
# Keep only id, name, artists, lyrics and rename for clarity.
REQUIRED = ['id', 'name', 'artists', 'lyrics']
missing  = [c for c in REQUIRED if c not in df.columns]
if missing:
    raise ValueError(f'Missing columns in CSV: {missing}')

df = df[REQUIRED].copy()
df = df.rename(columns={'name': 'title', 'artists': 'artist'})
print(f'After column selection  : {len(df):,} rows')

After column selection  : 955,320 rows


In [ ]:
# ── Step 3 · Remove songs with missing or very short lyrics
# Remove empty lyrics, and songs with fewer than 30 words
df = df[df['lyrics'].notna()].copy()
df['_word_count'] = df['lyrics'].astype(str).str.split().str.len()
df = df[df['_word_count'] >= 30].copy()
print(f'After null/short filter : {len(df):,} rows')

After null/short filter : 947,404 rows


In [ ]:
# ── Step 4 · Keep English-language songs only
# Use ASCII-ratio heuristic: keep songs where >=90% of characters are ASCII
df['_ascii'] = df['lyrics'].astype(str).map(ascii_ratio)
df = df[df['_ascii'] >= 0.90].copy()
print(f'After English filter    : {len(df):,} rows')

After English filter    : 889,697 rows


In [ ]:
# ── Step 5 · Clean up song titles and artist names (part 1/5)
# Define a stricter title normalizer to handle edge-cases (punctuation-only titles, mixed quotes, hyphens, etc.)
def normalize_title_strong(s: str) -> str:
    if not isinstance(s, str):
        return ''
    t = s.lower()
    # 1) Trim leading/trailing sequences of non-alphanumeric characters
    t = re.sub(r'^[^A-Za-z0-9]+|[^A-Za-z0-9]+$', '', t)
    # 2) Replace any remaining non-alphanumeric (except space) with a single space
    t = re.sub(r'[^A-Za-z0-9 ]+', ' ', t)
    # 3) Collapse multiple spaces and strip
    t = re.sub(r'\s+', ' ', t).strip()
    return t

# Keep the original, gentler normalizer for reference (used elsewhere if needed)
def normalize_title(s: str) -> str:
    if not isinstance(s, str):
        return ''
    t = s.lower()
    t = re.sub(r'^[^A-Za-z0-9]+|[^A-Za-z0-9]+$', '', t)   # trim non-alphanum at edges
    t = re.sub(r'[^A-Za-z0-9 ]+', ' ', t)                 # replace non-alphanum (except space) with space
    t = re.sub(r'\s+', ' ', t).strip()                   # collapse spaces and trim
    return t


In [ ]:
# ── Step 5 · Clean up song titles and artist names (part 2/5)
# Apply the stronger title normalizer
df['normalized_title'] = df['title'].astype(str).map(normalize_title_strong)

# Show a small sample of before -> after so we can inspect the changes
print('Sample original -> normalized (first 10 rows):')
print(df[['title', 'normalized_title']].head(10).to_string(index=False))

# Identify rows with invalid normalized titles (do not drop here) — helpful for later decisions
mask_invalid_title = (
    df['normalized_title'].isna()
    | (df['normalized_title'].str.len() < 2)
    | (df['normalized_title'].str.strip() == '')
    | (df['normalized_title'].str.fullmatch(r'\d+'))
)
num_invalid_titles = int(mask_invalid_title.sum())
print(f'Invalid normalized titles detected: {num_invalid_titles}')


Sample original -> normalized (first 10 rows):
                        title            normalized_title
                            !                            
                           !!                            
               !!De Repente!!                  de repente
               !!De Repente!!                  de repente
          !!Noble Stabbings!!             noble stabbings
               !I'll Be Back!                i ll be back
                       !Lost!                        lost
    !Que Vida! - Mono Version       que vida mono version
!Viva el Mal Viva el Capital! viva el mal viva el capital
                   !liaF cipE                   liaf cipe
Invalid normalized titles detected: 1770


In [ ]:
# ── Step 5 · Clean up song titles and artist names (part 3/5)
# 1) Clean `artist` column per requirements: lowercase, remove brackets [], remove single quotes, normalize whitespace
df['artist'] = df['artist'].astype(str).str.lower()
df['artist'] = df['artist'].str.replace(r'\[|\]', '', regex=True)
df['artist'] = df['artist'].str.replace("'", '', regex=False)
df['artist'] = df['artist'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Quick sample to verify artist cleaning
print('Artist cleaning sample:')
print(df[['artist']].head(10).to_string(index=False))


Artist cleaning sample:
          artist
        hellyeah
         yxngxr1
         rosendo
         rosendo
  dillinger four
           rilès
           rilès
            love
  elektroduendes
one morning left


In [ ]:
# ── Step 5 · Clean up song titles and artist names (part 4/5)
# 3) Create `normalized_artist`: lowercase, remove non-alphanumeric (except spaces), normalize whitespace
df['normalized_artist'] = df['artist'].astype(str).str.lower()
df['normalized_artist'] = df['normalized_artist'].str.replace(r'[^A-Za-z0-9 ]+', ' ', regex=True)
df['normalized_artist'] = df['normalized_artist'].str.replace(r'\s+', ' ', regex=True).str.strip()

# 4) Clean `lyrics` (preserve line breaks): reuse helper `normalize_lyrics` to lowercase, remove most punctuation but keep apostrophes, normalize spaces per line
df['lyrics'] = df['lyrics'].astype(str).map(normalize_lyrics)

# 5) Create `normalized_lyrics` for dedup: lowercase, remove punctuation, collapse ALL whitespace to single spaces (no line breaks)
df['normalized_lyrics'] = df['lyrics'].astype(str).str.lower()
df['normalized_lyrics'] = df['normalized_lyrics'].str.replace(r'[^\w\s]', ' ', regex=True)
df['normalized_lyrics'] = df['normalized_lyrics'].str.replace(r'\s+', ' ', regex=True).str.strip()


In [ ]:
# ── Step 5 · Clean up song titles and artist names (part 5/5)
# 6) Reorder columns exactly as specified (do not drop rows or deduplicate here)
cols_target = ['id', 'title', 'artist', 'normalized_title', 'normalized_artist', 'lyrics', 'normalized_lyrics']
# Ensure all target columns exist before reordering
for c in cols_target:
    if c not in df.columns:
        df[c] = ''
df = df[cols_target].copy()

# 9) Print preview: show all target columns in the requested order
preview = df.copy()
preview['lyrics_preview'] = preview['lyrics'].astype(str).str.slice(0, 100)
print('\nPreview (first 10 rows) — columns: ' + ','.join(cols_target))
# Display the exact requested columns: id, title (original), artist (cleaned), normalized_title, normalized_artist, lyrics, normalized_lyrics
print(preview[cols_target].head(10).to_string(index=False))



Preview (first 10 rows) — columns: id,title,artist,normalized_title,normalized_artist,lyrics,normalized_lyrics
                    id                         title           artist            normalized_title normalized_artist                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [ ]:
# ── Step 6 · Remove perfectly identical rows
before = len(df)
df = df.drop_duplicates().copy()
print(f'Removed exact duplicates        : {before - len(df):,}  →  {len(df):,} remain')

Removed exact duplicates        : 0  →  889,697 remain


In [ ]:
# ── Step 7 · Remove duplicates that are the same song by a different name
# Use composite key of normalized_title + normalized_artist
# Ensure the normalized fields exist (they were created in the title-cleaning step)
if 'normalized_title' not in df.columns:
    df['normalized_title'] = df['title'].astype(str).map(normalize_title)
if 'normalized_artist' not in df.columns:
    df['normalized_artist'] = df['artist'].astype(str).map(normalize_meta)

# Build composite key and drop duplicates keeping the first occurrence
df['_comp_key'] = df['normalized_title'].fillna('') + '_' + df['normalized_artist'].fillna('')
before = len(df)
df = df.drop_duplicates(subset=['_comp_key']).copy()
print(f'Removed composite-key duplicates: {before - len(df):,}  →  {len(df):,} remain')

Removed composite-key duplicates: 110,110  →  779,587 remain


In [ ]:
# ── Step 8 · Clean lyrics text and remove songs with identical lyrics
# Normalize lyrics (preserve line breaks) and use MD5 to fingerprint
df['_lyrics_norm'] = df['lyrics'].astype(str).map(normalize_lyrics)
df['_lyrics_hash'] = df['_lyrics_norm'].map(md5_hash)
before = len(df)
df = df.drop_duplicates(subset=['_lyrics_hash']).copy()
print(f'Removed duplicate lyrics (hash) : {before - len(df):,}  →  {len(df):,} remain')

Removed duplicate lyrics (hash) : 34,549  →  745,038 remain


In [ ]:
# Safety step: ensure `_word_count` exists before Step 9
# Some earlier reordering may have removed helper columns; recompute from `lyrics` if missing.
if '_word_count' not in df.columns:
    df['_word_count'] = df['lyrics'].astype(str).str.split().str.len()
else:
    # coerce to int if present
    df['_word_count'] = df['_word_count'].astype(int)
print('(_word_count) sample:', df['_word_count'].head(5).tolist())


(_word_count) sample: [472, 269, 112, 211, 523]


In [ ]:
# ── Step 9 · Remove songs that are unusually short or absurdly long
wc      = df['_word_count']
q1, q3  = wc.quantile(0.25), wc.quantile(0.75)
iqr     = q3 - q1
lo      = max(1, int(q1 - 1.5 * iqr))
hi      = int(q3 + 1.5 * iqr)
before  = len(df)
df = df[(df['_word_count'] >= lo) & (df['_word_count'] <= hi)].copy()
print(f'IQR word-count bounds           : [{lo}, {hi}]')
print(f'Removed outliers                : {before - len(df):,}  →  {len(df):,} remain')

IQR word-count bounds           : [1, 573]
Removed outliers                : 44,558  →  700,480 remain


In [ ]:
# ── Step 10 · Save the clean dataset (full schema with normalized helpers)
# Ensure the processed schema is saved exactly as required
cols_target = ['id', 'title', 'artist', 'normalized_title', 'normalized_artist', 'lyrics', 'normalized_lyrics']
for c in cols_target:
    if c not in df.columns:
        df[c] = ''
final_df = df[cols_target].copy()
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
final_df.to_csv(OUT_CSV, index=False)
print(f'\nSaved → {OUT_CSV}')
print(f'Final row count : {len(final_df):,}')
final_df.head(5)

In [ ]:
# ── Final · Display all notebook cells (reads the .ipynb and prints sources)
import json
nb_path = APP_ROOT / 'notebooks' / 'spotify960k_preprocessing.ipynb'
if nb_path.exists():
    with open(nb_path, 'r', encoding='utf-8') as fh:
        nb = json.load(fh)
    print(f'Notebook: {nb_path} - total cells: {len(nb.get("cells", []))}')
    for i, cell in enumerate(nb.get('cells', []), start=1):
        ctype = cell.get('cell_type', '<unknown>')
        src = ''.join(cell.get('source', []))
        print('\n' + '='*80)
        print(f'Cell {i} — type: {ctype}')
        print('-' * 40)
        print(src)
    # Also display the saved processed CSV preview (if present)
    try:
        if OUT_CSV.exists():
            print('\n' + '='*80)
            print('Saved processed CSV preview:')
            df_check = pd.read_csv(OUT_CSV)
            cols = ['id', 'title', 'artist', 'normalized_title', 'normalized_artist', 'lyrics', 'normalized_lyrics']
            for c in cols:
                if c not in df_check.columns:
                    df_check[c] = ''
            print(df_check[cols].head(10).to_string(index=False))
    except Exception as e:
        print('Could not read/display saved CSV:', e)
else:
    print('Notebook file not found:', nb_path)


Notebook file not found: C:\Users\Aman Tewari\Documents\aaa-college\sem6\SLA-pvt\approot\notebooks\spotify960k_preprocessing.ipynb
